# كشف الشذوذ بمرمّز ذاتي

**خطأ إعادة البناء بوصفه مسافة** · معالج رسوميات اختياري · ~25 دقيقة · Colab

لديك خمسة آلاف تسجيل لنبضات القلب، ولا تكاد تملك تسميات للحالات الشاذة، لأن الشذوذ هو بالضبط ما لم يجمع أحد ما يكفي منه. هذه هي الصورة المعتادة لكشف الشذوذ: الفئة النادرة نادرة في بيانات التدريب أيضاً، فلا يجد المصنّف ما يتعلمه. درّب نموذجاً على الحالة الطبيعية وحدها، ودع عجزه عن إعادة بناء ما لم يألفه يكون هو الإشارة.

### الهدف

درّب مرمّزاً ذاتياً على النبضات الطبيعية وحدها، واختر عتبة من توزيع خطأ إعادة البناء لا بالتخمين، وحقّق 0.90 على الأقل في مقياس F1 على مجموعة اختبار تضمّ الفئتين.

### الأوراق وراء هذه الورشة

- [deep-autoencoders](https://azimuth.blog/ar/paper/deep-autoencoders) — الفكرة القائلة إن طبقة ضيقة تُجبر النموذج على الاحتفاظ بما يهم فقط — هينتون وسالاخوتدينوف، 2006
- [isolation-forest](https://azimuth.blog/ar/paper/isolation-forest) — الأساس المرجعي الذي يعزل الشذوذ بدل أن ينمذج الحالة الطبيعية — ليو وتينغ وتشو، 2008

> احفظ نسخة في Drive قبل أن تبدأ (ملف ← حفظ نسخة في Drive). التعديلات على الأصل لا تُحفظ.

## الإعداد

`PROFILE` هو المقبض الوحيد للحجم. المستوى المجاني هو الافتراضي ويعمل داخل حدود Colab المجانية.

In [ ]:
SLUG = "anomaly-detection-autoencoder"
LANG = "ar"
PROFILE = "free"  # free | a100

# Colab defaults to inline figures; CI does not. Being explicit means the
# captured plot on the site and the plot you see are produced the same way.
%matplotlib inline

# The shim lives in this repository, not on PyPI: a workshop should never
# depend on a package index staying up.
import subprocess
import sys
from pathlib import Path

# Both steps are guarded. Colab users re-run the setup cell constantly,
# and an unguarded %cd descends one level EVERY time — which is how a
# second run ends up in azimuth-workshops/azimuth-workshops/... and the
# shim quietly resolves against the wrong tree.
REPO = 'azimuth-workshops'
if Path.cwd().name != REPO:
    if not Path(REPO).exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', 'stable',
             'https://github.com/MotazSabri/azimuth-workshops.git', REPO],
            check=True,
        )
    %cd azimuth-workshops

# Absolute, so it survives any later change of working directory.
sys.path.insert(0, str(Path.cwd() / 'shim'))

يحتاج المصنّف إلى أمثلة من كل فئة سيقابلها. وكشف الشذوذ هو الحالة التي لا يمكنك فيها امتلاكها: الفئة المهمة نادرة بحكم التعريف، والأمثلة التي بحوزتك لا تمثّل تلك التي لم ترها بعد. لذا نقلب المسألة. درّب نموذجاً على مهمة سهلة — أن ينسخ مدخله إلى مخرجه — لكن أجبره على المرور بطبقة أضيق من أن تنسخ كل شيء. سينفق تلك الميزانية على ما هو شائع. ثم أطعمه ما ليس شائعاً، وراقب فشله.

_فحص مسبق: تُفحص البيئة ويُتحقق من البيانات قبل أي تدريب._

In [ ]:
import azimuth_nb as azimuth

env = azimuth.setup(SLUG, lang=LANG, profile=PROFILE)

> **الورقة** · [deep-autoencoders](https://azimuth.blog/ar/paper/deep-autoencoders) — الفكرة القائلة إن طبقة ضيقة تُجبر النموذج على الاحتفاظ بما يهم فقط — هينتون وسالاخوتدينوف، 2006
>
> ورقة هينتون وسالاخوتدينوف لعام 2006 هي الموضع الذي تكفّ فيه الطبقة الضيقة عن كونها حيلة ضغط وتصير أداة لتعلّم التمثيل. والمرمّز أدناه هو الحجة ذاتها على نطاق أصغر بكثير.

_لاحظ التقسيم: مجموعة التدريب آثار طبيعية فقط. وهذه هي الطريقة كلها._

In [ ]:
import numpy as np

raw = np.loadtxt(env.assets["ecg5000.csv"], delimiter=",")
traces, labels = raw[:, :-1].astype(np.float32), raw[:, -1].astype(int)

# Min-max to [0, 1] using TRAINING statistics only. Fitting the scaler on
# everything would leak the abnormal range into the normal model — a quiet
# mistake that inflates every number downstream.
rng = np.random.default_rng(env.cfg["seed"])
order = rng.permutation(len(traces))
traces, labels = traces[order], labels[order]

split = int(0.8 * len(traces))
train_all, test_x = traces[:split], traces[split:]
train_labels, test_y = labels[:split], labels[split:]

# THE METHOD, IN ONE LINE: the model only ever sees normal traces.
train_x = train_all[train_labels == 1]

lo, hi = train_x.min(), train_x.max()
train_x = (train_x - lo) / (hi - lo)
test_x = (test_x - lo) / (hi - lo)

n_normal = int((test_y == 1).sum())
n_abnormal = int((test_y == 0).sum())
class_balance = {
    "trainNormal": len(train_x),
    "testNormal": n_normal,
    "testAbnormal": n_abnormal,
}

if env.lang == "ar":
    print(f"التدريب: {len(train_x)} أثراً طبيعياً فقط")
    print(f"الاختبار: {n_normal} طبيعي · {n_abnormal} شاذ")
else:
    print(f"train: {len(train_x)} normal traces only")
    print(f"test:  {n_normal} normal · {n_abnormal} abnormal")

> **الحجم** — كل ما يلي يقرأ أحجامه من `env.cfg` الآتية من الملف الذي اخترته في الأعلى. في المستوى المجاني هذا يعني 60 دورة وعنق زجاجة بثمانية أبعاد — وهو ما يكفي لفصل الفئتين فصلاً نظيفاً في نحو أربع دقائق.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(env.cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_features = train_x.shape[1]


class Autoencoder(nn.Module):
    """Symmetric encoder/decoder around a deliberately narrow layer.

    The bottleneck width is the entire experiment. Widen it and the model
    learns the identity function and reconstructs abnormal traces just as well
    as normal ones — at which point there is no detector left, only a copier.
    """

    def __init__(self, n_features: int, hidden: list[int], latent: int):
        super().__init__()
        widths = [n_features, *hidden]

        encoder: list[nn.Module] = []
        for a, b in zip(widths[:-1], widths[1:]):
            encoder += [nn.Linear(a, b), nn.ReLU()]
        encoder += [nn.Linear(widths[-1], latent), nn.ReLU()]
        self.encoder = nn.Sequential(*encoder)

        decoder: list[nn.Module] = []
        rev = [latent, *hidden[::-1]]
        for a, b in zip(rev[:-1], rev[1:]):
            decoder += [nn.Linear(a, b), nn.ReLU()]
        # Sigmoid because the inputs were scaled to [0, 1]: the output range
        # should be able to reach the input range and no further.
        decoder += [nn.Linear(rev[-1], n_features), nn.Sigmoid()]
        self.decoder = nn.Sequential(*decoder)

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = Autoencoder(n_features, list(env.cfg["hidden"]), env.cfg["latentDim"]).to(device)
param_count = sum(p.numel() for p in model.parameters())

env.explain("bottleneck")
if env.lang == "ar":
    print(f"عنق الزجاجة: {env.cfg['latentDim']} من أصل {n_features} بُعداً · {param_count:,} معامل")
else:
    print(f"bottleneck: {env.cfg['latentDim']} of {n_features} dims · {param_count:,} parameters")

_راقب الشكل لا الرقم الأخير وحده. تهبط الخسارة بحدة ثم تستقر على هضبة لعشرين أو ثلاثين دورة ثم تعاود الهبوط — عند تلك النقطة يكون النموذج قد كفّ عن تحسين الأثر المتوسط وبدأ يلائم التفاصيل التي كان يعمّمها. أوقفه عند الهضبة تشحن كاشفاً نصف مدرَّب يبدو وكأنه اكتمل."_

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_t = torch.from_numpy(train_x).to(device)
loader = DataLoader(
    TensorDataset(train_t, train_t),
    batch_size=env.cfg["batchSize"],
    shuffle=True,
)

optimizer = torch.optim.Adam(model.parameters(), lr=env.cfg["learningRate"])
criterion = nn.MSELoss()

loss_curve = []
model.train()
for epoch in range(env.cfg["epochs"]):
    total = 0.0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(batch_x)
    epoch_loss = total / len(train_t)
    loss_curve.append(epoch_loss)
    if epoch % 10 == 0 or epoch == env.cfg["epochs"] - 1:
        print(f"epoch {epoch:3d}  loss {epoch_loss:.5f}")

final_loss = loss_curve[-1]

_الصورة التي وُجدت الورشة كلها من أجلها: توزيعان والفجوة بينهما._

In [ ]:
import matplotlib.pyplot as plt


def reconstruction_error(x: np.ndarray) -> np.ndarray:
    """Mean absolute error per trace — one number for each recording.

    L1 rather than L2 on purpose: squared error lets a single badly-missed
    sample dominate a trace's score, which makes the threshold sensitive to
    noise rather than to shape.
    """
    model.eval()
    with torch.no_grad():
        t = torch.from_numpy(x).to(device)
        return torch.mean(torch.abs(model(t) - t), dim=1).cpu().numpy()


train_errors = reconstruction_error(train_x)
test_errors = reconstruction_error(test_x)

normal_errors = test_errors[test_y == 1]
abnormal_errors = test_errors[test_y == 0]
separation = float(abnormal_errors.mean() / normal_errors.mean())

env.explain("reconstruction error")
fig, ax = plt.subplots(figsize=(7, 3.6))
bins = np.linspace(0, float(np.percentile(test_errors, 99.5)), 60)
ax.hist(normal_errors, bins=bins, alpha=0.75, label="normal", color="#2a9d8f")
ax.hist(abnormal_errors, bins=bins, alpha=0.75, label="abnormal", color="#e76f51")
ax.set_xlabel("reconstruction error (MAE)")
ax.set_ylabel("traces")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

print(f"normal mean   {normal_errors.mean():.4f}")
print(f"abnormal mean {abnormal_errors.mean():.4f}")
separation_ok = env.check("separation", separation)

### تمرين — choose-threshold

اختر العتبة. الحركة البديهية أن تجرّب القيم حتى تبلغ F1 ذروتها — لكن هذا يستخدم تسميات الشذوذ التي لن تملكها يوم النشر. استخدم أخطاء التدريب بدلاً من ذلك: اختر مئيناً من الأخطاء على بيانات رآها النموذج فعلاً، وبرّر الرقم الذي اخترته.

_تلميح متاح: `env.hint(2)`_

In [ ]:
# YOUR TURN.
#
# Pick a percentile of `train_errors` — the errors on traces the model was
# trained on. Anything you can compute from this array is available on the day
# you deploy, with no abnormal examples in hand. Anything you compute from
# `abnormal_errors` is not.
#
# Start here and change the number, with a reason:
PERCENTILE = 95

threshold = float(np.percentile(train_errors, PERCENTILE))
print(f"threshold = {threshold:.4f}  (p{PERCENTILE} of training error)")

In [ ]:
predicted_abnormal = test_errors > threshold
actually_abnormal = test_y == 0

tp = int((predicted_abnormal & actually_abnormal).sum())
fp = int((predicted_abnormal & ~actually_abnormal).sum())
fn = int((~predicted_abnormal & actually_abnormal).sum())
tn = int((~predicted_abnormal & ~actually_abnormal).sum())

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
confusion = {"tp": tp, "fp": fp, "fn": fn, "tn": tn}

if env.lang == "ar":
    print(f"الدقة {precision:.3f} · الاستدعاء {recall:.3f} · F1 {f1:.3f}")
    print(f"أخطأ في {fn} حالة شاذة، وأنذر زوراً في {fp} حالة طبيعية")
else:
    print(f"precision {precision:.3f} · recall {recall:.3f} · F1 {f1:.3f}")
    print(f"missed {fn} abnormal · false-alarmed on {fp} normal")

f1_ok = env.check("f1", f1)

> **الورقة** · [isolation-forest](https://azimuth.blog/ar/paper/isolation-forest) — الأساس المرجعي الذي يعزل الشذوذ بدل أن ينمذج الحالة الطبيعية — ليو وتينغ وتشو، 2008
>
> من المفيد أن تعرف بمن تقارن. غابة العزل لا تنمذج الحالة الطبيعية أصلاً — بل تعزل النقاط بتقسيم عشوائي، على أساس أن الشذوذ يحتاج قطوعاً أقل لفصله. على هذه البيانات تقترب من نتيجتنا. أما حين يكون لـ«الطبيعي» بنية تستحق التعلّم، فلا.

_المقارنة هي المقصد لا الفوز. اقرأ أيهما كنت ستشحن._

In [ ]:
from sklearn.ensemble import IsolationForest

forest = IsolationForest(
    n_estimators=100,
    contamination=n_abnormal / len(test_y),
    random_state=env.cfg["seed"],
)
forest.fit(train_x)
forest_abnormal = forest.predict(test_x) == -1

b_tp = int((forest_abnormal & actually_abnormal).sum())
b_fp = int((forest_abnormal & ~actually_abnormal).sum())
b_fn = int((~forest_abnormal & actually_abnormal).sum())
b_precision = b_tp / (b_tp + b_fp) if (b_tp + b_fp) else 0.0
b_recall = b_tp / (b_tp + b_fn) if (b_tp + b_fn) else 0.0
baseline_f1 = (
    2 * b_precision * b_recall / (b_precision + b_recall) if (b_precision + b_recall) else 0.0
)

print(f"isolation forest F1 {baseline_f1:.3f}   ·   autoencoder F1 {f1:.3f}")

_الإيصال: رمز إتمامك والأرقام التي اشتُقّ منها._

In [ ]:
receipt = env.receipt()